In [ ]:
from src.preprocessamento.limpador_texto import LimpadorTexto, limpar_texto

limpador = LimpadorTexto()
print('Módulo carregado com sucesso.')

In [ ]:
# Texto de exemplo com todos os tipos de ruído presentes em um currículo real
trecho = (
    'João Silva | joao.silva@email.com | (11) 93206-5400 | https://linkedin.com/in/joao\n'
    'Experiência com node.js, c# e asp.net. Conhecimento em ml e nlp. '
    'Responsável pela administração dos servidores e infraestrutura de redes.'
)

print('TEXTO ORIGINAL:')
print(trecho)

In [ ]:
# Etapa 1 — Remoção de URLs
t1 = limpador.remover_urls(trecho)
print('Após remover URLs:')
print(t1)

In [ ]:
# Etapa 2 — Remoção de e-mails
t2 = limpador.remover_emails(t1)
print('Após remover e-mails:')
print(t2)

In [ ]:
# Etapa 3 — Remoção de telefones
t3 = limpador.remover_telefones(t2)
print('Após remover telefones:')
print(t3)

In [ ]:
# Etapa 4 — Normalização de termos técnicos
t4 = limpador.normalizar_termos_tecnicos(t3)
print('Após normalizar termos técnicos:')
print(t4)
print()

from src.preprocessamento.limpador_texto import NORMALIZACOES
print('Normalizações configuradas:')
exemplos = [
    ('node.js',  'nodejs'),
    ('c#',       'csharp'),
    ('c++',      'cplusplus'),
    ('asp.net',  'aspnet'),
    ('.net',     'dotnet'),
    ('ml',       'machinelearning'),
    ('ai',       'artificialintelligence'),
    ('nlp',      'naturallanguageprocessing'),
    ('gcp',      'googlecloudplatform'),
]
for antes, depois in exemplos:
    print(f'  {antes:<12} → {depois}')

In [ ]:
# Etapa 5 — Remoção de caracteres especiais
t5 = limpador.remover_caracteres_especiais(t4)
print('Após remover caracteres especiais:')
print(t5)

In [ ]:
# Etapa 6 — Remoção de acentos
t6 = limpador.remover_acentos(t5)
print('Após remover acentos:')
print(t6)

In [ ]:
# Etapa 7 — Tokenização, lematização e remoção de stopwords via spaCy
tokens = limpador.tokenizar_e_limpar(t6)
print(f'Tokens após lematização e remoção de stopwords ({len(tokens)} tokens):')
print(tokens)
print()

from src.preprocessamento.limpador_texto import STOPWORDS_DOMINIO
palavras_t6 = t6.lower().split()
removidas = [p for p in palavras_t6 if p in STOPWORDS_DOMINIO]
print('Stopwords de domínio removidas neste texto:')
print(removidas if removidas else '(nenhuma neste trecho)')

In [ ]:
texto_limpo = limpar_texto(texto)

print('TEXTO BRUTO (primeiros 400 caracteres):')
print(texto[:400])
print()
print('TEXTO APÓS LIMPEZA COMPLETA (primeiros 400 caracteres):')
print(texto_limpo[:400])
print()
print(f'Caracteres antes : {len(texto)}')
print(f'Caracteres depois: {len(texto_limpo)}')
print(f'Redução          : {round((1 - len(texto_limpo)/len(texto)) * 100, 1)}%')

In [ ]:
# Demonstra por que a lematização é importante para o matching:
# palavras diferentes de um mesmo verbo/substantivo são tratadas como iguais
import spacy
nlp = spacy.load('pt_core_news_sm')

exemplos_lematizacao = [
    'desenvolvidas', 'desenvolvendo', 'desenvolveu',
    'experiências', 'experiência',
    'trabalhando', 'trabalhou', 'trabalhos',
    'gerenciando', 'gerenciou', 'gerenciamento',
]

print('Efeito da lematização — diferentes formas → mesma raiz:')
print(f'{"Palavra original":<22} → {"Lema (forma base)"}')
print('-' * 45)
for palavra in exemplos_lematizacao:
    doc = nlp(palavra)
    lema = doc[0].lemma_.lower()
    print(f'{palavra:<22} → {lema}')

In [ ]:
# Demonstra a diferença que o pré-processamento faz no matching:
# mesmo conteúdo escrito de formas diferentes
curriculo_v1 = 'Desenvolvedor com experiência em Node.js, C# e ASP.NET. Conhecimento em ML e NLP.'
curriculo_v2 = 'Programador nodejs csharp aspnet machinelearning naturallanguageprocessing'

limpo_v1 = limpar_texto(curriculo_v1)
limpo_v2 = limpar_texto(curriculo_v2)

print('Texto 1 (original):  ', curriculo_v1)
print('Texto 1 (limpo):     ', limpo_v1)
print()
print('Texto 2 (original):  ', curriculo_v2)
print('Texto 2 (limpo):     ', limpo_v2)
print()

# Calcula a similaridade ANTES e DEPOIS da limpeza
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer()

# Sem limpeza
mat_bruto = vec.fit_transform([curriculo_v1, curriculo_v2])
score_bruto = cosine_similarity(mat_bruto[0:1], mat_bruto[1:2])[0][0]

# Com limpeza
mat_limpo = vec.fit_transform([limpo_v1, limpo_v2])
score_limpo = cosine_similarity(mat_limpo[0:1], mat_limpo[1:2])[0][0]

print(f'Similaridade SEM pré-processamento: {score_bruto:.1%}')
print(f'Similaridade COM pré-processamento: {score_limpo:.1%}')
print()
print('Os dois textos descrevem o mesmo perfil mas escritos de formas diferentes.')
print('O pré-processamento permite que o sistema reconheça essa equivalência.')